<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day03_practice2_%EC%97%AD%EC%A0%84%ED%8C%8C_%ED%95%B4%EB%B6%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [2]:
x = torch.tensor(1.0)
y = torch.tensor(1.0)

w1 = torch.tensor(0.5, requires_grad=True)
b1 = torch.tensor(0.1, requires_grad=True)
w2 = torch.tensor(0.8, requires_grad=True)
b2 = torch.tensor(0.2, requires_grad=True)

In [3]:
# 셀 2. 순전파(forward)
# x ──[w1, b1]──> z1 ──σ──> h ──[w2, b2]──> z2 ──σ──> ŷ

z1 = w1 * x + b1          # 선형
h = torch.sigmoid(z1)     # 은닉 출력
z2 = w2 * h + b2          # 선형
y_hat = torch.sigmoid(z2) # 최종 출력(예측)
L = (y_hat - y) ** 2      #손실

print("--- 순전파 ---")
print(f"z1={z1.item():.4f} h={h.item():.4f} z2={z2.item():.4f}")
print(f"예측 ŷ={y_hat.item():.4f} 정답 y=1.0 손실 L={L.item():.4f}")

--- 순전파 ---
z1=0.6000 h=0.6457 z2=0.7165
예측 ŷ=0.6718 정답 y=1.0 손실 L=0.1077


In [4]:
# 셀 3. 역전파 손계산 - 체인 룰
# 목표: L 을 줄이려면 w2, w1 을 어느 방향으로 움직여야 하나? = ∂L/∂w
#
# 체인 룰: 합성 함수의 미분은 "사슬처럼 이어 곱한다"
#
#   ∂L/∂w2 = ∂L/∂ŷ · ∂ŷ/∂z2 · ∂z2/∂w2
#             ─┬──   ─┬───    ─┬───
#          2(ŷ-y)   ŷ(1-ŷ)      h        ← 시그모이드 미분: σ' = σ(1-σ)
#
#   ∂L/∂w1 = ∂L/∂ŷ · ∂ŷ/∂z2 · ∂z2/∂h · ∂h/∂z1 · ∂z1/∂w1
#          = 2(ŷ-y) · ŷ(1-ŷ) ·   w2   · h(1-h) ·   x
#
#  뒤(출력)에서 앞(입력)으로, 오차의 책임을 나눠 전달 — 그래서 '역'전파.

with torch.no_grad():             # 손계산이므로 추적 없이
  dL_dyhat = 2 * (y_hat - y)      # ∂L/∂ŷ
  dyhat_dz2 = y_hat * (1 - y_hat) # σ'(z2) Sigmoid 미분
  dh_dz1 = h * (1 - h)            # σ'(z1) Sigmoid 미분

  grad_w2_hand = dL_dyhat * dyhat_dz2 * h
  grad_b2_hand = dL_dyhat * dyhat_dz2 * 1
  grad_w1_hand = dL_dyhat * dyhat_dz2 * w2 * dh_dz1 * x
  grad_b1_hand = dL_dyhat * dyhat_dz2 * w2 * dh_dz1 * 1

print("--- 손계산 (체인 룰) ---")
print(f"∂L/∂w2 = {grad_w2_hand.item():.6f}")
print(f"∂L/∂w1 = {grad_w1_hand.item():.6f}")

--- 손계산 (체인 룰) ---
∂L/∂w2 = -0.093426
∂L/∂w1 = -0.026484


In [5]:
# 셀 4. autograd 검증 - backward()가 내는 값과 비교
L.backward()

print("--- autograd (backward) ---")
print(f"∂L/∂w2 = {w2.grad.item():.6f}")
print(f"∂L/∂w1 = {w1.grad.item():.6f}")

# 일치 확인 (오차 범위 내 동일하면 통과)
assert torch.allclose(grad_w2_hand, w2.grad), "w2 불일치"
assert torch.allclose(grad_w1_hand, w1.grad), "w1 불일치"
assert torch.allclose(grad_b2_hand, b2.grad), "b2 불일치"
assert torch.allclose(grad_b1_hand, b1.grad), "b1 불일치"
print("\n 손계산 = autograd 완전 일치")

--- autograd (backward) ---
∂L/∂w2 = -0.093426
∂L/∂w1 = -0.026484

 손계산 = autograd 완전 일치


In [6]:
# 셀 5. 기울기 소실 - 시그모이드를 깊게 쌓으면

import torch.nn as nn

def first_layer_grad(activation, depth=10):
  layers = []
  for _ in range(depth):
    layers += [nn.Linear(8,8), activation()]
  layers += [nn.Linear(8,1)]
  net = nn.Sequential(*layers)
  x = torch.randn(16,8)
  loss = net(x).mean() # 정의된 신경망 net에 입력 데이터 x를 통과시켜 예측 결과를 얻고, 이 예측 결과의 평균을 계산하여 손실(loss)로 설정
  loss.backward()
  return net[0].weight.grad.abs().mean().item() # 첫 층 기울기의 평균 크기

torch.manual_seed(0)
g_sig = first_layer_grad(nn.Sigmoid)
torch.manual_seed(0)
g_relu = first_layer_grad(nn.ReLU)

print(f"Sigmoid 10층: {g_sig:.10f}") # 기울기 소실
print(f"ReLU 10층: {g_relu:.10f}")

# 현대 신경망의 은닉층은 대부분 ReLU 계열, 시그모이드는 '마지막 출력(확률)'에서만

Sigmoid 10층: 0.0000000003
ReLU 10층: 0.0000012067
